In [ ]:
import duckdb
import time
from pathlib import Path

DB_PATH = "./mimiciv.duckdb"
MIMIC_PATH = Path(
    "physionet.org/files/mimiciv/3.1"
)  # adjust if your raw files live elsewhere

HOSP = MIMIC_PATH / "hosp"
ICU = MIMIC_PATH / "icu"

TABLES = {
    # Hosp module
    "hosp.patients": HOSP / "patients.csv.gz",
    "hosp.admissions": HOSP / "admissions.csv.gz",
    "hosp.transfers": HOSP / "transfers.csv.gz",
    "hosp.diagnoses_icd": HOSP / "diagnoses_icd.csv.gz",
    "hosp.d_icd_diagnoses": HOSP / "d_icd_diagnoses.csv.gz",
    "hosp.procedures_icd": HOSP / "procedures_icd.csv.gz",
    "hosp.d_icd_procedures": HOSP / "d_icd_procedures.csv.gz",
    "hosp.labevents": HOSP / "labevents.csv.gz",
    "hosp.d_labitems": HOSP / "d_labitems.csv.gz",
    "hosp.prescriptions": HOSP / "prescriptions.csv.gz",
    "hosp.microbiologyevents": HOSP / "microbiologyevents.csv.gz",
    "hosp.services": HOSP / "services.csv.gz",
    "hosp.drgcodes": HOSP / "drgcodes.csv.gz",
    "hosp.omr": HOSP / "omr.csv.gz",
    # ICU module
    "icu.icustays": ICU / "icustays.csv.gz",
    "icu.chartevents": ICU / "chartevents.csv.gz",
    "icu.d_items": ICU / "d_items.csv.gz",
    "icu.inputevents": ICU / "inputevents.csv.gz",
    "icu.outputevents": ICU / "outputevents.csv.gz",
    "icu.procedureevents": ICU / "procedureevents.csv.gz",
    "icu.datetimeevents": ICU / "datetimeevents.csv.gz",
}


def load_tables(con):
    con.execute("CREATE SCHEMA IF NOT EXISTS hosp")
    con.execute("CREATE SCHEMA IF NOT EXISTS icu")
    for table_name, file_path in TABLES.items():
        if not file_path.exists():
            print(f"  SKIP {table_name}: file not found at {file_path}")
            continue
        print(f"  Loading {table_name}...", end=" ", flush=True)
        start = time.time()
        con.execute(f"DROP TABLE IF EXISTS {table_name}")
        con.execute(f"""
            CREATE TABLE {table_name} AS
            SELECT * FROM read_csv_auto('{file_path}', header=True)
        """)
        n_rows = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        elapsed = time.time() - start
        print(f"{n_rows:,} rows in {elapsed:.1f}s")


def create_views(con):
    # Joins patients with admissions; adds hospital LOS in hours
    con.execute("""
        CREATE OR REPLACE VIEW patient_admissions AS
        SELECT
            p.subject_id, p.gender, p.anchor_age, p.anchor_year, p.dod,
            a.hadm_id, a.admittime, a.dischtime, a.deathtime,
            a.admission_type, a.admission_location, a.discharge_location,
            a.insurance, a.language, a.marital_status, a.race,
            a.hospital_expire_flag,
            EXTRACT(EPOCH FROM (a.dischtime - a.admittime)) / 3600 AS hospital_los_hours
        FROM hosp.patients p
        JOIN hosp.admissions a ON p.subject_id = a.subject_id
    """)

    # Enriches ICU stays with demographics from patient_admissions; adds ICU LOS in hours
    con.execute("""
        CREATE OR REPLACE VIEW icu_stays_full AS
        SELECT
            i.stay_id, i.subject_id, i.hadm_id,
            i.first_careunit, i.last_careunit, i.intime, i.outtime,
            EXTRACT(EPOCH FROM (i.outtime - i.intime)) / 3600 AS icu_los_hours,
            pa.gender, pa.anchor_age, pa.race, pa.insurance,
            pa.admission_type, pa.hospital_expire_flag, pa.dod
        FROM icu.icustays i
        JOIN patient_admissions pa ON i.hadm_id = pa.hadm_id
    """)

    # Filters chartevents to CAM/delirium assessments only; normalises values to binary 0/1
    con.execute("""
        CREATE OR REPLACE VIEW cam_icu_assessments AS
        SELECT
            ce.subject_id, ce.hadm_id, ce.stay_id, ce.charttime, ce.itemid,
            di.label AS item_label, ce.value, ce.valuenum,
            CASE 
                WHEN ce.value IN ('Positive', 'Yes', 'positive', 'yes') THEN 1
                WHEN ce.value IN ('Negative', 'No', 'negative', 'no') THEN 0
                ELSE NULL
            END AS cam_positive
        FROM icu.chartevents ce
        JOIN icu.d_items di ON ce.itemid = di.itemid
        WHERE LOWER(di.label) LIKE '%cam%'
           OR LOWER(di.label) LIKE '%delirium%'
           OR LOWER(di.label) LIKE '%confusion assessment%'
    """)

    # Aggregates CAM assessments per ICU stay: total/positive counts, ever-delirious flag, first/last assessment times
    con.execute("""
        CREATE OR REPLACE VIEW delirium_summary AS
        SELECT
            stay_id,
            COUNT(*) AS total_assessments,
            SUM(cam_positive) AS positive_assessments,
            MAX(cam_positive) AS ever_delirious,
            MIN(CASE WHEN cam_positive = 1 THEN charttime END) AS first_positive_time,
            MIN(charttime) AS first_assessment_time,
            MAX(charttime) AS last_assessment_time
        FROM cam_icu_assessments
        WHERE cam_positive IS NOT NULL
        GROUP BY stay_id
    """)

    # Joins ICD diagnosis codes with their human-readable long titles
    con.execute("""
        CREATE OR REPLACE VIEW diagnoses_labeled AS
        SELECT
            d.subject_id, d.hadm_id, d.seq_num, d.icd_code, d.icd_version,
            dd.long_title AS diagnosis
        FROM hosp.diagnoses_icd d
        LEFT JOIN hosp.d_icd_diagnoses dd 
            ON d.icd_code = dd.icd_code AND d.icd_version = dd.icd_version
    """)

    # Joins lab results with item labels and categories for human-readable lab names and units
    con.execute("""
        CREATE OR REPLACE VIEW labs_labeled AS
        SELECT
            le.subject_id, le.hadm_id, le.charttime, le.itemid,
            dl.label AS lab_name, dl.category AS lab_category,
            le.value, le.valuenum, le.valueuom AS unit, le.flag
        FROM hosp.labevents le
        JOIN hosp.d_labitems dl ON le.itemid = dl.itemid
    """)


# Build only if the database doesn't exist; otherwise just connect
if Path(DB_PATH).exists():
    print(f"Database already exists at {DB_PATH}, connecting...")
    con = duckdb.connect(database=DB_PATH)
else:
    print(f"No database found at {DB_PATH}, building from raw files...")
    if not MIMIC_PATH.exists():
        raise FileNotFoundError(
            f"Raw MIMIC-IV files not found at {MIMIC_PATH}. "
            f"Update MIMIC_PATH at the top of this cell to point to your data."
        )
    con = duckdb.connect(database=DB_PATH)
    print("\n>>> Loading tables...")
    load_tables(con)
    print("\n>>> Creating views...")
    create_views(con)
    print(f"\nDatabase built at {DB_PATH}")

print("Connection ready.")

Database already exists at ./mimiciv.duckdb, connecting...
Connection ready.


In [4]:
con.execute("""
    SELECT COUNT(*) AS icu_stays_under_18
    FROM icu_stays_full
    WHERE anchor_age < 18
""").fetchdf()

,icu_stays_under_18
0,0


In [7]:
invalid = con.execute("""
    SELECT stay_id
    FROM icu.icustays
    WHERE intime IS NULL
       OR outtime IS NULL
       OR outtime <= intime
""").fetchdf()

print(f"Deleting {len(invalid)} invalid rows...")

con.execute("""
    DELETE FROM icu.icustays
    WHERE intime IS NULL
       OR outtime IS NULL
       OR outtime <= intime
""")

print(f"Deleted {len(invalid)} rows from icu.icustays")

Deleting 0 invalid rows...
Deleted 0 rows from icu.icustays


In [2]:
con.execute("""
    SELECT
        ever_delirious,
        COUNT(*) AS n_stays
    FROM delirium_summary
    GROUP BY ever_delirious
    ORDER BY ever_delirious
""").df()

,ever_delirious,n_stays
0,0,39046
1,1,33403


In [11]:
# ── icu_stays_full ───────────────────────────────────────────────────────────

print("=" * 60)
print("VIEW: icu_stays_full")
print("=" * 60)

total = con.execute("SELECT COUNT(*) FROM icu_stays_full").fetchone()[0]
print(f"  Total rows: {total:,}\n")

r = con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE stay_id IS NULL)                     AS null_stay_id,
        COUNT(*) FILTER (WHERE subject_id IS NULL)                  AS null_subject_id,
        COUNT(*) FILTER (WHERE hadm_id IS NULL)                     AS null_hadm_id,
        COUNT(*) FILTER (WHERE intime IS NULL)                      AS null_intime,
        COUNT(*) FILTER (WHERE outtime IS NULL)                     AS null_outtime,
        COUNT(*) FILTER (WHERE outtime <= intime)                   AS invalid_los,
        COUNT(*) FILTER (WHERE icu_los_hours < 0)                   AS negative_los,
        COUNT(*) FILTER (WHERE icu_los_hours > 720)                 AS los_over_30d,
        COUNT(*) FILTER (WHERE anchor_age < 18)                     AS underage,
        COUNT(*) FILTER (WHERE anchor_age > 120)                    AS invalid_age,
        COUNT(*) FILTER (WHERE anchor_age IS NULL)                  AS null_age,
        COUNT(*) FILTER (WHERE gender IS NULL)                      AS null_gender,
        COUNT(*) FILTER (WHERE gender NOT IN ('M', 'F'))            AS invalid_gender,
        COUNT(*) FILTER (WHERE race IS NULL)                        AS null_race,
        COUNT(*) FILTER (WHERE insurance IS NULL)                   AS null_insurance,
        COUNT(*) FILTER (WHERE first_careunit IS NULL)              AS null_first_careunit,
        COUNT(*) FILTER (WHERE last_careunit IS NULL)               AS null_last_careunit,
        COUNT(*) FILTER (WHERE hospital_expire_flag NOT IN (0, 1))  AS invalid_expire_flag,
    FROM icu_stays_full
""").fetchdf().iloc[0]

issues = {k: int(v) for k, v in r.items() if int(v) > 0}
if issues:
    for check, count in issues.items():
        print(f"  {check:<35} {count:>10,}")
else:
    print("  ✓ No issues found")

# -- Deletion candidates for icu_stays_full --
# con.execute("""
#     DELETE FROM icu.icustays
#     WHERE stay_id IN (
#         SELECT stay_id FROM icu_stays_full
#         WHERE intime IS NULL
#            OR outtime IS NULL
#            OR outtime <= intime
#            OR anchor_age < 18
#     )
# """)


# ── cam_icu_assessments ──────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("VIEW: cam_icu_assessments")
print("=" * 60)

total = con.execute("SELECT COUNT(*) FROM cam_icu_assessments").fetchone()[0]
print(f"  Total rows: {total:,}\n")

r = con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE stay_id IS NULL)                     AS null_stay_id,
        COUNT(*) FILTER (WHERE subject_id IS NULL)                  AS null_subject_id,
        COUNT(*) FILTER (WHERE hadm_id IS NULL)                     AS null_hadm_id,
        COUNT(*) FILTER (WHERE charttime IS NULL)                   AS null_charttime,
        COUNT(*) FILTER (WHERE itemid IS NULL)                      AS null_itemid,
        COUNT(*) FILTER (WHERE item_label IS NULL)                  AS null_item_label,
        COUNT(*) FILTER (WHERE value IS NULL)                       AS null_value,
        COUNT(*) FILTER (WHERE cam_positive IS NULL)                AS unresolved_cam,
        COUNT(DISTINCT stay_id)
            FILTER (WHERE cam_positive IS NOT NULL)                 AS stays_with_valid_cam,
    FROM cam_icu_assessments
""").fetchdf().iloc[0]

issues = {k: int(v) for k, v in r.items() if int(v) > 0}
if issues:
    for check, count in issues.items():
        print(f"  {check:<35} {count:>10,}")
else:
    print("  ✓ No issues found")

print("\n  Distinct cam_positive values:")
print(con.execute("""
    SELECT cam_positive, COUNT(*) AS n
    FROM cam_icu_assessments
    GROUP BY cam_positive
    ORDER BY cam_positive
""").fetchdf().to_string(index=False))

# -- Deletion candidates for cam_icu_assessments --
# con.execute("""
#     DELETE FROM icu.chartevents
#     WHERE stay_id IN (
#         SELECT stay_id FROM cam_icu_assessments WHERE charttime IS NULL
#     )
#       AND itemid IN (SELECT itemid FROM cam_icu_assessments WHERE charttime IS NULL)
# """)


# ── delirium_summary ─────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("VIEW: delirium_summary")
print("=" * 60)

total = con.execute("SELECT COUNT(*) FROM delirium_summary").fetchone()[0]
print(f"  Total rows (unique stay_ids): {total:,}\n")

r = con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE stay_id IS NULL)                             AS null_stay_id,
        COUNT(*) FILTER (WHERE total_assessments IS NULL
                            OR total_assessments = 0)                       AS zero_assessments,
        COUNT(*) FILTER (WHERE positive_assessments IS NULL)                AS null_positive_count,
        COUNT(*) FILTER (WHERE positive_assessments > total_assessments)    AS positives_exceed_total,
        COUNT(*) FILTER (WHERE ever_delirious NOT IN (0, 1))                AS invalid_ever_delirious,
        COUNT(*) FILTER (WHERE first_positive_time IS NOT NULL
                            AND first_assessment_time IS NOT NULL
                            AND first_positive_time < first_assessment_time) AS positive_before_first,
        COUNT(*) FILTER (WHERE last_assessment_time < first_assessment_time) AS last_before_first,
        COUNT(*) FILTER (WHERE ever_delirious = 1
                            AND first_positive_time IS NULL)                AS delirious_no_first_time,
    FROM delirium_summary
""").fetchdf().iloc[0]

issues = {k: int(v) for k, v in r.items() if int(v) > 0}
if issues:
    for check, count in issues.items():
        print(f"  {check:<40} {count:>10,}")
else:
    print("  ✓ No issues found")

print("\n  Class distribution (ever_delirious):")
print(con.execute("""
    SELECT ever_delirious, COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM delirium_summary
    GROUP BY ever_delirious
    ORDER BY ever_delirious
""").fetchdf().to_string(index=False))

# -- Deletion candidates for delirium_summary --
# con.execute("""
#     DELETE FROM icu.chartevents
#     WHERE stay_id IN (
#         SELECT stay_id FROM delirium_summary
#         WHERE total_assessments = 0
#            OR positive_assessments > total_assessments
#     )
# """)


# ── labs_labeled ─────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("VIEW: labs_labeled")
print("=" * 60)

total = con.execute("SELECT COUNT(*) FROM labs_labeled").fetchone()[0]
print(f"  Total rows: {total:,}\n")

r = con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE subject_id IS NULL)          AS null_subject_id,
        COUNT(*) FILTER (WHERE hadm_id IS NULL)             AS null_hadm_id,
        COUNT(*) FILTER (WHERE charttime IS NULL)           AS null_charttime,
        COUNT(*) FILTER (WHERE itemid IS NULL)              AS null_itemid,
        COUNT(*) FILTER (WHERE lab_name IS NULL)            AS null_lab_name,
        COUNT(*) FILTER (WHERE lab_category IS NULL)        AS null_lab_category,
        COUNT(*) FILTER (WHERE value IS NULL
                            AND valuenum IS NULL)           AS both_values_null,
        COUNT(*) FILTER (WHERE valuenum < 0)                AS negative_valuenum,
        COUNT(*) FILTER (WHERE unit IS NULL)                AS null_unit,
    FROM labs_labeled
""").fetchdf().iloc[0]

issues = {k: int(v) for k, v in r.items() if int(v) > 0}
if issues:
    for check, count in issues.items():
        print(f"  {check:<35} {count:>10,}")
else:
    print("  ✓ No issues found")

print("\n  Implausible values for key labs:")
print(con.execute("""
    SELECT
        lab_name,
        itemid,
        COUNT(*)                        AS n,
        ROUND(MIN(valuenum), 2)         AS min_val,
        ROUND(MAX(valuenum), 2)         AS max_val,
        ROUND(AVG(valuenum), 2)         AS mean_val,
        COUNT(*) FILTER (WHERE
            (itemid = 50912 AND (valuenum < 0.1  OR valuenum > 50  ))
         OR (itemid = 50931 AND (valuenum < 10   OR valuenum > 2000))
         OR (itemid = 50983 AND (valuenum < 90   OR valuenum > 180 ))
         OR (itemid = 50971 AND (valuenum < 1    OR valuenum > 10  ))
         OR (itemid = 51222 AND (valuenum < 1    OR valuenum > 25  ))
         OR (itemid = 51301 AND (valuenum < 0.1  OR valuenum > 500 ))
         OR (itemid = 50813 AND (valuenum < 0.1  OR valuenum > 50  ))
         OR (itemid = 50820 AND (valuenum < 6.5  OR valuenum > 7.9 ))
        )                               AS implausible,
    FROM labs_labeled
    WHERE itemid IN (50912, 50931, 50983, 50971, 51222, 51301, 50813, 50820)
      AND valuenum IS NOT NULL
    GROUP BY itemid, lab_name
    ORDER BY itemid
""").fetchdf().to_string(index=False))

# -- Deletion candidates for labs_labeled --
# con.execute("""
#     DELETE FROM hosp.labevents
#     WHERE (value IS NULL AND valuenum IS NULL)
#        OR valuenum < 0
#        OR (itemid = 50912 AND (valuenum < 0.1 OR valuenum > 50))
#        OR (itemid = 50931 AND (valuenum < 10  OR valuenum > 2000))
#        OR (itemid = 50983 AND (valuenum < 90  OR valuenum > 180))
#        OR (itemid = 50971 AND (valuenum < 1   OR valuenum > 10))
#        OR (itemid = 51222 AND (valuenum < 1   OR valuenum > 25))
#        OR (itemid = 51301 AND (valuenum < 0.1 OR valuenum > 500))
#        OR (itemid = 50813 AND (valuenum < 0.1 OR valuenum > 50))
#        OR (itemid = 50820 AND (valuenum < 6.5 OR valuenum > 7.9))
# """)

VIEW: icu_stays_full
  Total rows: 94,444

  los_over_30d                               673
  null_insurance                           1,523

VIEW: cam_icu_assessments
  Total rows: 1,465,940

  unresolved_cam                         166,435
  stays_with_valid_cam                    72,449

  Distinct cam_positive values:
 cam_positive      n
            0 754181
            1 545324
         <NA> 166435

VIEW: delirium_summary
  Total rows (unique stay_ids): 72,449

  ✓ No issues found

  Class distribution (ever_delirious):
 ever_delirious     n  pct
              0 39046 53.9
              1 33403 46.1

VIEW: labs_labeled
  Total rows: 158,374,764

  null_hadm_id                        73,768,897
  both_values_null                    16,062,783
  negative_valuenum                      659,529
  null_unit                           23,522,996

  Implausible values for key labs:
         lab_name  itemid       n  min_val    max_val  mean_val  implausible
          Lactate   50813  6696